In [ ]:
import json
from datasets import Dataset

with open('../outputs/synthetic_logs.json', 'r') as f:
    logs = json.load(f)

print(f"Loaded {len(logs)} logs")

training_data = []

for log in logs:
    # Полный prompt со всеми полями
    prompt = f"""Analyze this employee security log and determine if the activity is NORMAL, SUSPICIOUS, or ANOMALY:

Timestamp: {log['timestamp']}
Employee ID: {log['employee_id']}
Session ID: {log['session_id']}
IP Address: {log['ip_address']}
User Agent: {log['user_agent']}
Action Type: {log['action_type']}
Resource Accessed: {log['resource_accessed']}
Resource Type: {log['resource_type']}
Request Status: {log['request_status']}
Data Size: {log['data_size_bytes']} bytes
Geolocation: {log['geolocation']}
Device Info: {log['device_info']}

Classification:"""
    
    response = f"{log['label']}\n\nExplanation: {log['explanation']}"
    
    training_data.append({
        "messages": [
            {"role": "system", "content": "You are an expert cybersecurity AI assistant specializing in employee behavior analysis and insider threat detection."},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
        ]
    })

output_file = '../outputs/prepared_dataset.json'
with open(output_file, 'w') as f:
    json.dump(training_data, f, indent=2)

print(f"✅ Prepared {len(training_data)} training examples")
print(f"Saved to {output_file}")

normal = sum(1 for d in training_data if 'NORMAL' in d['messages'][2]['content'])
suspicious = sum(1 for d in training_data if 'SUSPICIOUS' in d['messages'][2]['content'])
anomaly = sum(1 for d in training_data if 'ANOMALY' in d['messages'][2]['content'])

print(f"\nTraining data distribution:")
print(f"  - NORMAL: {normal}")
print(f"  - SUSPICIOUS: {suspicious}")
print(f"  - ANOMALY: {anomaly}")

print("\nExample training sample:")
print(json.dumps(training_data[0], indent=2))
